In [1]:
import sys
sys.path.append('../../Simulate/')
import importlib


import numpy as np
import subprocess
from typing import Dict
from UtilityFunctions import retrieve_iupac


from StreamHTSIM import StreamHTSIM
from LockedIterator import LockedIterator

In [2]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + 'data/ref/BSB_test.fa'
ref_fasta_allA  = working_path + 'data/ref/all_A.fa'
ref_fasta_chr21 = working_path + 'data/ref/chr21.fa'
ref_fasta_empty = working_path + 'data/ref/empty.fa'
ref_fasta_nonexist = working_path + 'data/ref/nonexist.fa'

# Code to test for different situation 

In [ ]:
htsim_arg= ['/home/wbguo/iproject/BSReadSim/HTSIM/htsim']

sim_dict = {'-N':1000, 
            '-h':0, 
            '-T':0,
            '-1':100, 
            '-2':100,
            '-e':0.005,
            '-i':400,
            '-I':25,
            '-r':0.001, 
            '-R':0.15,
            '-X':0.15, 
            '-A':0.05,
            '-f':1,
           }

In [ ]:
sim_sub = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val]

sim_cmd =  sim_sub + [ref_fasta]
sim_cmd_allA = sim_sub + [ref_fasta_allA] 
sim_cmd_chr21= sim_sub + [ref_fasta_chr21] 
sim_cmd_empty_fasta = sim_sub + [ref_fasta_empty]
sim_cmd_nonexist_fasta = sim_sub + [ref_fasta_nonexist]

sim_dict['-r'] = 0
sim_cmd_no_snp  = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val] + [ref_fasta]
sim_dict['-r'] = 0.001

sim_dict['-N'] = 10
sim_cmd_small_N = htsim_arg + [str(item) for key_val in sim_dict.items() for item in key_val] + [ref_fasta]
sim_dict['-N'] = 1000

# No snp

In [ ]:
' '.join(sim_cmd_no_snp)

In [ ]:
i = 0
for variant_contig, sim_data in StreamHTSIM(sim_cmd_no_snp):
    if variant_contig:
        print(sim_data)
    
    if variant_contig:
        print(variant_contig)
    if not variant_contig:
        [sim_data[0]['read_id'], sim_data[0]['pair']]
        ++i

In [ ]:
sim_data[0]

# check size of each

In [ ]:
len(sim_data[0]['ctx'])

In [ ]:
import sys
print(sys.getsizeof(sim_data[0]))

In [ ]:
from pympler import asizeof
print(asizeof.asizeof(sim_data[0]))

In [ ]:
test_sim = StreamHTSIM(sim_cmd_no_snp)

In [ ]:
test_it = LockedIterator(test_sim)

In [ ]:
test_it

In [ ]:
next(test_it)

# save sim_data for ReadProcessor test

In [ ]:
sim_data2 = next(test_it)
sim_data2

In [ ]:
import pickle

read_pickle_file = working_path + "/pkl/read_pair_raw.pkl"
with open(read_pickle_file, 'wb') as FILE:
    pickle.dump(sim_data2, FILE)

# Test the running time and size of core steps

In [ ]:
' '.join(sim_cmd)

In [ ]:
htsim = subprocess.Popen(sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
sim_iter = iter(htsim.stdout.readline, b'')

In [ ]:
type(sim_iter)

In [ ]:
def get_line(sim_iter):
    try:
        line = next(sim_iter).strip()
    except StopIteration:
        print("End of output\n")
        return None
    else:
        return line

In [ ]:
def process_variant_line(line: str) -> Dict:
    line_split = line.split('\t')

    try:
        chrom, pos, ref, alt, heter_flag = line_split
    except ValueError:
        return dict(chrom=line_split[0], pos = None)
    else:
        heter = heter_flag == '+'
        indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
        offset= indel * max(len(ref), len(alt))
        if indel:
            iupac  = None
        else:
            iupac  = retrieve_iupac(alt)
            alt    = list(set(iupac) - set(ref))[0]
        return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                    offset=offset, heter=heter, indel=indel, iupac=iupac)


def collect_variants(sim_iter):
    variant_dict = {}

    while True:
        line = get_line(sim_iter)
        if line == 'Contig Variant End':
            return variant_info['chrom'], variant_dict

        variant_info = process_variant_line(line)
        if variant_info['pos']:
            assert variant_info['pos'] not in variant_dict
            variant_dict[variant_info['pos']] = variant_info

In [ ]:
v = collect_variants(sim_iter)

In [ ]:
v

In [ ]:
from pympler import asizeof
print(asizeof.asizeof(v))

In [ ]:
len(v[1])

In [ ]:
while True:
    x = get_line(sim_iter)
    if x[0] == "@":
        break

y = get_line(sim_iter)
z = get_line(sim_iter)
t = get_line(sim_iter)

In [ ]:
x

In [ ]:
x.split(' ')

In [ ]:
y

In [ ]:
z

In [ ]:
t

In [ ]:
np.frombuffer(t.encode('utf-8'), np.int8)

In [ ]:
#### 12.2 us & 2808 byte for a read, used the fputc for output
def process_read_name(line_list: list):
    '''parse read lines'''
    # header, seq, comment process
    read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line_list[0].split(' ')
    cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
    seq = np.frombuffer(line_list[1].encode(), dtype=np.int8)
    _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = np.frombuffer(line_list[3].encode(), np.int8)
    return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos),
                n_sub=int(n_sub), n_indel=int(n_indel),
                insert_size=int(insert_size), inner_dist=int(inner_dist),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
%timeit process_read_name([x,y,z,t]) 

In [ ]:
import sys
obj = process_read_name([x,y,z,t])
sys.getsizeof(obj)

In [ ]:
asizeof.asizeof(obj)

In [ ]:
#### 12.6 us & 2856 byte for a read, used the fputc for output
def process_read_name2(line_list: list):
    '''parse read lines'''
    # header, seq, comment process
    read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line_list[0].split(' ')
    cgr = np.frombuffer(bytearray(cgr.encode()), dtype=np.int8)
    seq = np.frombuffer(bytearray(line_list[1].encode()), dtype=np.int8)
    _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = np.frombuffer(line_list[3].encode(), np.int8)
    return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos),
                n_sub=int(n_sub), n_indel=int(n_indel),
                insert_size=int(insert_size), inner_dist=int(inner_dist),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
%timeit process_read_name2([x,y,z,t]) 

In [ ]:
obj2 =process_read_name2([x,y,z,t]) 
sys.getsizeof(obj2)

In [ ]:
asizeof.asizeof(obj2)

In [ ]:
obj

In [ ]:
chr(obj['qual'])

In [ ]:
''.join(['ACGT'[i] for i in obj['seq']])

In [ ]:
while True:
    x1 = get_line(sim_iter)
    if x1[0] == "@":
        break

y1 = get_line(sim_iter)
z1 = get_line(sim_iter)
t1 = get_line(sim_iter)

In [ ]:
obj1 = process_read_name([x1,y1,z1,t1])

In [ ]:
obj1

In [ ]:
sim_data = [obj, obj1]

In [ ]:
sys.getsizeof(sim_data)

In [ ]:
print(asizeof.asizeof(sim_data))

In [ ]:
%timeit obj1['start'] + np.arange(len(obj1['seq']))

In [ ]:
x = obj1['start'] + np.arange(len(obj1['seq']))

In [ ]:
%timeit x + obj1['ofs']

# Speed & memory test result

In [ ]:
#### 25 us & 640 byte for a read, used the %d for output
ascii_idx = np.array([i for i in range(48,58)] + [i for i in range(97, 103)])
ascii_val = np.array([i for i in range(0,16)])
ascii_arr = np.full(127, -1).astype(np.int8)
ascii_arr[ascii_idx] = ascii_val

def process_read_name2(line_list: list, read_len: int):
    read_id, pair, flag_pos, flag_mut, flag_indel, cgr = line_list[0].split(' ')
    cgr = np.bitwise_and(np.frombuffer(cgr.encode(), dtype=np.int8), 0x03)
    seq = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, start, end, cover_pos, n_sub, n_indel, insrt_len, insrt_len2, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id=read_id, pair=int(pair), flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos), n_sub=int(n_sub), n_indel=int(n_indel), 
                insrt_len=int(insrt_len), insrt_len2=int(insrt_len2),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
#### 38 us & 360 byte for a read, used the %d for output
def process_read_name3(line_list: list, read_len: int):
    arr = np.full([4, read_len], np.NaN)
    read_id, pair, num_var, num_indel, start, end, mut = line_list[0].split(' ')
    arr[0] = np.bitwise_and(np.frombuffer(mut.encode(), dtype=np.int8), 0x03)
    arr[1] = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, n_sub, n_indel, insrt_len, ofs = line_list[2].split(':')
    arr[2] = np.fromstring(ofs, dtype=np.int8, sep = ',')
    arr[3] = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id = read_id, pair = int(pair),  start=int(start), end=int(end), 
                num_var = int(num_var), num_indel = int(num_indel), n_sub = int(n_sub), n_indel = int(n_indel),
                arr = arr)

In [ ]:
### if use %d
%timeit np.fromstring(y, dtype=np.int8)                               # 1.56 us, will give 48-51
%timeit np.frombuffer(y.encode(), dtype=np.int8)                      # 0.9  us, will give 48-51
%timeit np.array(list(y), dtype=np.int8)                              # 13.5 us, will give 0-3
%timeit np.frombuffer(y.encode(), dtype=np.int8) - 48                 # 4.25 us, will give 0-3
%timeit np.bitwise_and(np.frombuffer(y.encode(), dtype=np.int8), 0x3) # 4.38 us, will give 0-3
%timeit ascii_arr[np.frombuffer(y.encode(), dtype=np.int8)]           # 4.8  us, will give 0-3